In [1]:
import os, sys, shutil, subprocess, stat

# helper function
def exists(path):
    val = os.path.exists(path)
    if val:
        print(f'{path} already exits. Using cached. Delete it manually to recieve it again!')
    return val

def _rm_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)

# Only download requirements.txt if not already cached
if not exists('requirements.txt'):
    if os.path.exists('req'):
        shutil.rmtree('req', onerror=_rm_readonly)
    subprocess.run(['git', 'clone', 'https://gist.github.com/dgedon/8a7b91714568dc35d0527233e9ceada4.git', 'req'], check=True)
    shutil.copy('req/requirements.txt', 'requirements.txt')
    shutil.rmtree('req', onerror=_rm_readonly)

# Read requirements.txt and remove all version constraints
with open('requirements.txt', 'r') as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if '==' in line:
        new_lines.append(line.split('==')[0] + '\n')
    else:
        new_lines.append(line)

with open('requirements.txt', 'w') as f:
    f.writelines(new_lines)

print('Modified requirements.txt: removed all version constraints.')


requirements.txt already exits. Using cached. Delete it manually to recieve it again!
Modified requirements.txt: removed all version constraints.


In [2]:
# Import
import torch
import torch.nn as nn
import numpy as np
from tqdm.notebook import trange, tqdm
import h5py
import pandas as pd
import matplotlib.pyplot as plt
import ast
%matplotlib inline


In [13]:
set_data = 'L'  # L = large, S = small

# set seed
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

# choose variables
"""
TASK: Adapt the following hyperparameters. Using defaults from Signal based code
"""
iter= 10
# PARAMETERS 

learning_rate = 1e-3
weight_decay = 0.05
num_epochs = 80
batch_size = 64

---
## The data set


In [5]:
classes = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

if set_data == 'S':
    h5_names = ['signal_test_s.h5', 'signal_train_s.h5']
    txt_names = ['Test_RECORDS_LowRes_s.txt', 'Train_RECORDS_LowRes_s.txt']
else:
    h5_names = ['signal_test_l.h5', 'signal_train_l.h5']
    txt_names = ['Test_RECORDS_LowRes.txt', 'Train_RECORDS_LowRes.txt']




path = r'..\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\\'
sampling_rate=100

txt_file = path + "RECORDS_LowRes.txt"
with open(txt_file, "r", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
#X = load_raw_data(Y, sampling_rate, path)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

# Split data into train and test
test_fold = 10
# Train
#X_train = X[np.where(Y.strat_fold != test_fold)]
y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass
# Test
#X_test = X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

y_test = y_test.iloc[0:].to_list()
y_train = y_train.iloc[0:].to_list()

for i in range(len(y_test)):
    lst = np.zeros(len(classes))
    z = 0
    for j in classes:
        if j in y_test[i]:
            lst[z] = 1
        z+=1
    y_test[i] = lst

for i in range(len(y_train)):
    lst = np.zeros(len(classes))
    z = 0
    for j in classes:
        if j in y_train[i]:
            lst[z] = 1
        z+=1
    y_train[i] = lst

y_test = np.stack(y_test)
y_train = np.stack(y_train)

print(y_train.shape)

# y_test = y_test[:arrays_test.shape[0]]
# y_train = y_train[:arrays_train.shape[0]]

#SELECT DATA SIZE ====================================================================
data_size = 0
if set_data == 'S':
    data_size = 2000
    print('Using small dataset...')
elif set_data == 'L':
    data_size = len(lines)
    print('Using large dataset...')
else:
    print('Error: No dataset specified')
    raise SystemExit  # Stops execution
# ===================================================================================

ratio = len(y_test)/len(lines)
print(f'Division of test data vs train data: {ratio}')
data_size_test = int(ratio*data_size)
data_size_train = data_size - data_size_test



lines_test = []
lines_train = []
for i in range(len(lines)):
    if Y.strat_fold.iloc[i] == test_fold:
        lines_test.append(lines[i])
    else:
        lines_train.append(lines[i])

y_test = y_test[:data_size_test]
y_train = y_train[:data_size_train]

# lines_test = lines_test[:data_size_test]
# lines_train = lines_train[:data_size_train]




if not exists(path + txt_names[0]):

    with open(path + txt_names[0], "w", encoding="utf-8") as f:
        f.write("\n".join(lines_test))
        
if not exists(path + txt_names[1]):

    with open(path + txt_names[1], "w", encoding="utf-8") as f:
        f.write("\n".join(lines_train))





# Generate test
def _generate_h5(records_txt, out_h5):
    """Run generate_h5.py, raising on failure. Removes corrupt output on error."""
    result = subprocess.run(
        [sys.executable, 'ecg-preprocessing-main/generate_h5.py',
         '--new_freq', '400', '--new_len', '4096',
         '--remove_baseline', '--use_all_leads',
         records_txt, out_h5],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        if os.path.exists(out_h5):
            os.remove(out_h5)  # remove corrupt partial file
        raise RuntimeError(f"generate_h5.py failed: {result.stderr}")

# Generate test H5
if not exists(h5_names[0]):
    _generate_h5(path + txt_names[0], h5_names[0])

with h5py.File(h5_names[0], "r+") as h5f:
    if "labels" not in h5f:
        h5f.create_dataset("labels", data=y_test, compression="gzip")

# Generate train H5
if not exists(h5_names[1]):
    _generate_h5(path + txt_names[1], h5_names[1])

with h5py.File(h5_names[1], "r+") as h5f:
    if "labels" not in h5f:
        h5f.create_dataset("labels", data=y_train, compression="gzip")




(19601, 5)
Using large dataset...
Division of test data vs train data: 0.10083031331712464
..\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\\Test_RECORDS_LowRes.txt already exits. Using cached. Delete it manually to recieve it again!
..\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\\Train_RECORDS_LowRes.txt already exits. Using cached. Delete it manually to recieve it again!
signal_test_l.h5 already exits. Using cached. Delete it manually to recieve it again!
signal_train_l.h5 already exits. Using cached. Delete it manually to recieve it again!


In [9]:
class Model(nn.Module):
    def __init__(self,):
        super(Model, self).__init__()
        self.kernel_size = 3

        # conv layer 1
        downsample1 = self._downsample(4096, 128)
        self.conv1 = nn.Conv1d(in_channels=8,
                               out_channels=64,
                               kernel_size=self.kernel_size,
                               stride=downsample1,
                               padding=self._padding(downsample1),
                               bias=False)
        self.bn1 = nn.BatchNorm1d(64)

        # conv layer 2
        downsample2 = self._downsample(128, 64)
        self.conv2 = nn.Conv1d(in_channels=64,
                               out_channels=128,
                               kernel_size=self.kernel_size,
                               stride=downsample2,
                               padding=self._padding(downsample2),
                               bias=False)
        self.bn2 = nn.BatchNorm1d(128)

        # conv layer 3
        downsample3 = self._downsample(64, 32)
        self.conv3 = nn.Conv1d(in_channels=128,
                               out_channels=256,
                               kernel_size=self.kernel_size,
                               stride=downsample3,
                               padding=self._padding(downsample3),
                               bias=False)
        self.bn3 = nn.BatchNorm1d(256)

        # conv layer 4
        downsample4 = self._downsample(32, 16)
        self.conv4 = nn.Conv1d(in_channels=256,
                               out_channels=512,
                               kernel_size=self.kernel_size,
                               stride=downsample4,
                               padding=self._padding(downsample4),
                               bias=False)
        self.bn4 = nn.BatchNorm1d(512)

        # conv layer 5
        downsample5 = self._downsample(16, 8)
        self.conv5 = nn.Conv1d(in_channels=512,
                               out_channels=1024,
                               kernel_size=self.kernel_size,
                               stride=downsample5,
                               padding=self._padding(downsample5),
                               bias=False)
        self.bn5 = nn.BatchNorm1d(1024)

        # linear layer 1
        self.lin1 = nn.Linear(in_features=1024*8,
                             out_features=512*8)

        # linear layer 2
        self.lin2 = nn.Linear(in_features=512*8,
                             out_features=1)

        # ReLU
        self.relu = nn.ReLU()

        # Dropout
        self.drop = nn.Dropout(p=0.2)

    def _padding(self, downsample):
        return max(0, int(np.floor((self.kernel_size - downsample + 1) / 2)))

    def _downsample(self, seq_len_in, seq_len_out):
        return int(seq_len_in // seq_len_out)


    def forward(self, x):
        x = x.transpose(2,1)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.relu(self.bn5(self.conv5(x)))

        x_flat = x.view(x.size(0), -1)
        x = self.lin1(x_flat)
        x = self.relu(x)
        x = self.drop(x)
        x = self.lin2(x)

        return x

In [10]:
def train_loop(epoch, dataloader, model, optimizer, loss_function, device):
    # model to training mode (important to correctly handle dropout or batchnorm layers)
    model.train()
    # allocation
    total_loss = 0  # accumulated loss
    n_entries = 0   # accumulated number of data points
    # progress bar def
    train_pbar = tqdm(dataloader, desc="Training Epoch {epoch:2d}".format(epoch=epoch), leave=True)
    # training loop
    for traces, labels in train_pbar:
        # data to device (CPU or GPU if available)
        traces, labels = traces.to(device), labels.to(device)

        """
        TASK: Insert your code here. This task can be done in 5 lines of code.
        """
        optimizer.zero_grad()
        outputs = model(traces)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        # Update accumulated values
        total_loss += loss.detach().cpu().numpy()
        n_entries += len(traces)

        # Update progress bar
        train_pbar.set_postfix({'loss': total_loss / n_entries})
    train_pbar.close()
    return total_loss / n_entries

In [11]:
def eval_loop(epoch, dataloader, model, loss_function, device):
    # model to evaluation mode (important to correctly handle dropout or batchnorm layers)
    model.eval()
    # allocation
    total_loss = 0  # accumulated loss
    n_entries = 0   # accumulated number of data points
    valid_probs = []  # accumulated predicted probabilities
    valid_true = [] # accumulated true labels

    # progress bar def
    eval_pbar = tqdm(dataloader, desc="Evaluation Epoch {epoch:2d}".format(epoch=epoch), leave=True)
    # evaluation loop
    for traces_cpu, labels_cpu in eval_pbar:
        # data to device (CPU or GPU if available)
        traces, labels = traces_cpu.to(device), labels_cpu.to(device)

        """
        TASK: Insert your code here. This task can be done in 6 lines of code.
        """

        with torch.no_grad():
            outputs = model(traces)
            loss = loss_function(outputs, labels)
            probabilities = torch.sigmoid(outputs)

        valid_probs.append(probabilities.cpu().numpy())
        valid_true.append(labels.cpu().numpy())

        # Update accumulated values
        total_loss += loss.detach().cpu().numpy()
        n_entries += len(traces)

        # Update progress bar
        eval_pbar.set_postfix({'loss': total_loss / n_entries})
    eval_pbar.close()
    return total_loss / n_entries, np.vstack(valid_probs), np.vstack(valid_true)

In [12]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device count:", torch.cuda.device_count())
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))
    x = torch.randn(2, 2).to("cuda")
    print("Tensor device:", x.device)

CUDA available: True
Device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 3060 Ti
Tensor device: cuda:0


In [14]:
from torch.utils.data import TensorDataset, random_split, DataLoader

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tqdm.write("Use device: {device:}\n".format(device=device))

# =============== Build data loaders ======================================#
tqdm.write("Building data loaders...")

# path_to_h5_train, path_to_csv_train, path_to_records = 'codesubset/train.h5', 'codesubset/train.csv', 'codesubset/train/RECORDS.txt'
# # load traces
# traces = torch.tensor(h5py.File(path_to_h5_train, 'r')['tracings'][()], dtype=torch.float32)
# # load labels
# ids_traces = [int(x.split('TNMG')[1]) for x in list(pd.read_csv(path_to_records, header=None)[0])] # Get order of ids in traces
# df = pd.read_csv(path_to_csv_train)
# df.set_index('id_exam', inplace=True)
# df = df.reindex(ids_traces) # make sure the order is the same
# labels = torch.tensor(np.array(df['AF']), dtype=torch.float32).reshape(-1,1)

if set_data == 'S':
    path_to_h5_train = 'signal_train_s.h5'
else:
    path_to_h5_train = 'signal_train_l.h5'



traces = torch.tensor(h5py.File(path_to_h5_train, 'r')['tracings'][()], dtype=torch.float32)
labels = torch.tensor(h5py.File(path_to_h5_train, 'r')['labels'][()], dtype=torch.float32) 


# load dataset
dataset = TensorDataset(traces, labels)
len_dataset = len(dataset)
dataset_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
n_classes = len(torch.unique(labels))
# split data
"""
TASK: Split the dataset in train and validation; Insert your code here.
This can be done in <=4 line of code
"""
train_size = int(0.8 * len_dataset)
valid_size = len_dataset - train_size
dataset_train, dataset_valid = random_split(dataset, [train_size, valid_size])


# build data loaders
train_dataloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(dataset_valid, batch_size=batch_size, shuffle=False)


valid_indices = dataset_valid.indices
valid_traces, valid_true = traces[valid_indices], labels[valid_indices]
print(valid_traces.shape, valid_true.shape)
tqdm.write("Done!\n")

Use device: cuda

Building data loaders...
torch.Size([3921, 4096, 12]) torch.Size([3921, 5])
Done!



In [15]:
# =============== Define model ============================================#
tqdm.write("Define model...")
"""
TASK: Replace the baseline model with your model; Insert your code here
"""

# Model parameters (Using defaults from program) ----------
seq_length, N_LEADS = traces.shape[1:] # seq_length: sample dim 4096 , N_LEADS:the 12 leads
kernel_size=17
dropout_rate = 0.8
net_filter_size= [64, 128, 196, 256, 320]
net_seq_lengh=[4096, 1024, 256, 64, 16]

#-----------------------------------------------------
N_CLASSES = len(classes)  # 5 diagnosis
model = ResNet1d(input_dim=(N_LEADS, seq_length),
                    blocks_dim=list(zip(net_filter_size, net_seq_lengh)),
                    n_classes=N_CLASSES,
                    kernel_size=kernel_size,
                    dropout_rate=dropout_rate)
model.to(device=device)
tqdm.write("Done!\n")



# =============== Define loss function ====================================#
"""
TASK: define the loss; Insert your code here. This can be done in 1 line of code
"""
# Define weights because imbalance
class_counts = np.array([9514, 5469, 5235, 4898, 2649])
N = np.sum(class_counts)

weights = N / class_counts
weights = weights / np.max(weights) # Normalize
weights = torch.tensor(weights, dtype=torch.float32).to(device)
print(f'Class weights: {weights}')

#weights2 = torch.tensor([1.291, 2.986, 3.164, 3.451, 7.229], dtype=torch.float32).to(device)
weights2 = (N-class_counts) / class_counts
weights2 = weights2 / np.max(weights2) # Normalize
weights2 = torch.tensor(weights2, dtype=torch.float32).to(device)
print(f'Class weights2: {weights2}')

tqdm.write("Define loss function...")
#loss_function = nn.CrossEntropyLoss()
loss_function = nn.BCEWithLogitsLoss() #pos_weight=weights2
#loss_function = nn.BCELoss()
tqdm.write("Done!\n")

# =============== Define optimizer ========================================#
tqdm.write("Define optimiser...")
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
tqdm.write("Done!\n")

# =============== Define lr scheduler =====================================#


lr_scheduler= torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

# Option C: plateau-based (use val loss)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=8, min_lr=1e-7
)
# lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
"""
OPTIONAL: define a learning rate scheduler; Insert your code here
"""
lr_scheduler = None

Define model...
Done!

Class weights: tensor([0.2784, 0.4844, 0.5060, 0.5408, 1.0000], device='cuda:0')
Class weights2: tensor([0.2023, 0.4300, 0.4539, 0.4924, 1.0000], device='cuda:0')
Define loss function...
Done!

Define optimiser...
Done!



## TRAINING 

In [16]:
os.makedirs('record_history', exist_ok=True)
os.makedirs('../../signal_MODEL', exist_ok=True)



# =============== Train model =============================================#
tqdm.write("Training...")
best_loss = np.inf
best_auroc = -np.inf
# allocation
train_loss_all, valid_loss_all = [], []

score_list = []
# loop over epochs
for epoch in trange(1, num_epochs + 1):
    # training loop
    train_loss = train_loop(epoch, train_dataloader, model, optimizer, loss_function, device)
    # validation loop
    valid_loss, y_pred, y_true = eval_loop(epoch, valid_dataloader, model, loss_function, device)

    # collect losses
    train_loss_all.append(train_loss)
    valid_loss_all.append(valid_loss)

    # compute validation metrics for performance evaluation
    """
    TASK: compute validation metrics (e.g. AUROC); Insert your code here
    This can be done e.g. in 5 lines of code
    """
    from sklearn.metrics import roc_auc_score
    # auroc = roc_auc_score(y_true=y_true, y_score=y_pred) # Previous-------------
    valid_auroc = roc_auc_score(y_true=y_true, y_score=y_pred, average='macro')# Now
    score_list.append(valid_auroc)

    # save best model: here we save the model only for the lowest validation loss
    # if valid_loss < best_loss: Previous
    if valid_auroc > best_auroc:
        # Save model parameters
        torch.save({'model': model.state_dict()}, 'signal_model.pth')
        torch.save({'model': model.state_dict()}, f'../../signal_MODEL/model_{iter}.pth')
        # Update best validation loss
        # best_loss = valid_loss  # Previous------------
        best_auroc = valid_auroc # Now -------------
        # statement
        model_save_state = "Best model -> saved"
    else:
        model_save_state = ""

    # Print message
    tqdm.write('Epoch {epoch:2d}: \t'
                'Train Loss {train_loss:.6f} \t'
                'Valid Loss {valid_loss:.6f} \t'
                '{model_save}'
                .format(epoch=epoch,
                        train_loss=train_loss,
                        valid_loss=valid_loss,
                        model_save=model_save_state)
                    )

    # Update learning rate with lr-scheduler
    
    if lr_scheduler:
        lr_scheduler.step(best_auroc) 
    
    diff_score=  np.array([best_auroc]*len(score_list)) - np.array(score_list)
    if len(score_list)>40 and all(x >0 for x in diff_score[-40:]):
        print("Early stopping criterion met. Stopping training.")
        break
"""
TASK: Here it can make sense to plot your learning curve; Insert your code here
"""


Training...


  0%|          | 0/80 [00:00<?, ?it/s]

Training Epoch  1:   0%|          | 0/245 [00:00<?, ?it/s]

Evaluation Epoch  1:   0%|          | 0/62 [00:00<?, ?it/s]

Epoch  1: 	Train Loss 0.007296 	Valid Loss 0.006184 	Best model -> saved


...

Epoch 79: 	Train Loss 0.002773 	Valid Loss 0.005296 	


Training Epoch 80:   0%|          | 0/245 [00:00<?, ?it/s]

Evaluation Epoch 80:   0%|          | 0/62 [00:00<?, ?it/s]

Epoch 80: 	Train Loss 0.002721 	Valid Loss 0.004773 	


'\nTASK: Here it can make sense to plot your learning curve; Insert your code here\n'

In [17]:
m_path = "record_history/"
metrics_names=['train_loss', 'valid_loss', 'valid_score']

mh5= h5py.File(f"record_history/metrics_signal.h5", "a")




if f"train_loss_{iter}" not in mh5.keys():
    mh5.create_dataset(f"train_loss_{iter}", data=np.array(train_loss_all))

if f"valid_loss_{iter}" not in mh5.keys():
    mh5.create_dataset(f"valid_loss_{iter}", data=np.array(valid_loss_all))

if f"valid_score_{iter}" not in mh5.keys():
    
    mh5.create_dataset(f"valid_score_{iter}", data=np.array(score_list))
    
mh5.close()

---
## Model Testing

Since we saved our best model, we can now load the trained model and make predictions on the test data set. We save the predictions in a csv file which will be uploaded as part of the deliverables. Note that we take a `Sigmoid()` function on the model prediction in order to obtain soft predictions (probabilities) instead of hard predictions (0s or 1s).

In [19]:
# build the dataloader once and re-use when running the cell below possibly multiple times.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============== Build data loaders ==========================================#
tqdm.write("Building data loaders...")
# load data
if set_data == 'S':
    path_to_h5_test = 'signal_test_s.h5'
else:
    path_to_h5_test = 'signal_test_l.h5'
    
traces_test = torch.tensor(h5py.File(path_to_h5_test, 'r')['tracings'][()], dtype=torch.float32)
labels_test = torch.tensor(h5py.File(path_to_h5_test, 'r')['labels'][()], dtype=torch.float32)
dataset_test = TensorDataset(traces_test, labels_test)
len_dataset_test = len(dataset_test)

# build data loaders
test_dataloader = DataLoader(dataset_test, batch_size=32, shuffle=False)




tqdm.write("Done!\n")

Building data loaders...
Done!



In [20]:
# =============== Select iteration to test ================================================#

iter=10


# load stored model parameters

ckpt = torch.load('signal_model.pth', map_location=lambda storage, loc: storage)
model.load_state_dict(ckpt['model'])
# put model on device
model.to(device=device)
tqdm.write("Done!\n")

# =============== Evaluate model ==============================================#
model.eval()
# allocation
test_pred = torch.zeros(len_dataset_test,len(classes))
test_true = torch.zeros(len_dataset_test,len(classes))
# progress bar def
test_pbar = tqdm(test_dataloader, desc="Testing")

# evaluation loop
end=0
for traces in test_pbar:
    # data to device
    labels = traces[1].to(device)
    traces = traces[0].to(device)
    start = end

    with torch.no_grad():
        # Forward pass
        model_output = model(traces)

        # store output
        end = min(start + len(model_output), test_pred.shape[0])
        test_pred[start:end] = torch.nn.Sigmoid()(model_output).detach().cpu()
        test_true[start:end] = labels

test_pbar.close()


# =============== Evaluate on validation ==============================================#
model.eval()
# allocation
valid_pred = torch.zeros(valid_size,len(classes))
valid_true = torch.zeros(valid_size,len(classes))

# progress bar def
valid_pbar = tqdm(valid_dataloader, desc="Validating")

# evaluation loop
end=0
for traces in valid_pbar:
    # data to device
    labels= traces[1].to(device)
    traces = traces[0].to(device)
    start = end
    
    with torch.no_grad():
        # Forward pass
        model_output = model(traces)

        # store output
        end = min(start + len(model_output), valid_pred.shape[0])
        valid_pred[start:end] = torch.nn.Sigmoid()(model_output).detach().cpu()
        valid_true[start:end] = labels

valid_pbar.close()



# =============== Evaluate on dataset ==============================================#
# model.eval()
# # allocation
# z_pred = torch.zeros(len_dataset,len(classes))
# data_true = torch.zeros(len_dataset,len(classes))
# # progress bar def
# train_pbar = tqdm(dataset_dataloader, desc="Data set evaluation")

# # evaluation loop
# end=0
# for traces in train_pbar:
#     # data to device
#     labels= traces[1].to(device)
#     traces = traces[0].to(device)
#     start = end
    
#     with torch.no_grad():
#         # Forward pass
#         model_output = model(traces)

#         # store output
#         end = min(start + len(model_output), z_pred.shape[0])
#         z_pred[start:end] = (model_output).detach().cpu()
#         data_true[start:end] = labels

# train_pbar.close()

# =============== Save predictions ============================================#
soft_pred = np.stack((1-test_pred.numpy(), test_pred.numpy()),axis=1).squeeze()


Done!



Testing:   0%|          | 0/69 [00:00<?, ?it/s]

Validating:   0%|          | 0/62 [00:00<?, ?it/s]

In [21]:
# ── Generate signal logits for Knowledge Distillation ────────────────────────
# Run the trained signal model on all training signals and store raw logits
# in train_l.h5 (or train_s.h5) so the image notebook can use them for KD training.
# Requires preprocess.py to have been run first to create the image H5 file.

set_data_kd = set_data  # uses same set_data ('L' or 'S') as above
path_sig_train = 'signal_train_l.h5' if set_data_kd == 'L' else 'signal_train_s.h5'
path_img_train = 'train_l.h5'        if set_data_kd == 'L' else 'train_s.h5'

if not os.path.exists(path_img_train):
    print(f'SKIP: {path_img_train} not found.')
    print('Run preprocess.py first:  python preprocess.py --dataset S  (or L)')
else:
    with h5py.File(path_img_train, 'r+') as img_h5:
        if 'signal_logits' in img_h5:
            print("signal_logits already present in", path_img_train)
        else:
            print("Loading training signals...")
            with h5py.File(path_sig_train, 'r') as sig_h5:
                traces_all = torch.tensor(sig_h5['tracings'][()], dtype=torch.float32)

            n_img = img_h5['images'].shape[0]
            traces_all = traces_all[:n_img]  # align to image count (same ecg_id order)

            print(f"Running signal model on {len(traces_all)} training samples...")
            model.eval()
            all_logits = []
            kd_batch = 64
            with torch.no_grad():
                for i in range(0, len(traces_all), kd_batch):
                    batch = traces_all[i:i + kd_batch].to(device)
                    logits = model(batch)
                    all_logits.append(logits.cpu().numpy())
            all_logits = np.vstack(all_logits)

            img_h5.create_dataset('signal_logits', data=all_logits, compression='gzip')
            print(f"signal_logits saved to {path_img_train}, shape: {all_logits.shape}")


signal_logits already present in train_l.h5


In [22]:
loss_function = nn.BCELoss()

from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss, brier_score_loss

def brier_score_per_class(y_true, y_pred_proba):
    """
    Calculate Brier score for each class separately
    """
    y_true = np.asarray(y_true)
    y_pred_proba = np.asarray(y_pred_proba)
    
    n_classes = y_true.shape[1]
    scores = []
    
    for k in range(n_classes):
        score = np.mean((y_pred_proba[:, k] - y_true[:, k]) ** 2)
        scores.append(score)
    
    return np.array(scores)

# Lower is better for these
lg_loss = loss_function(torch.tensor(test_pred), torch.tensor(test_true)).item()
# Also lower is better
brier = brier_score_per_class(y_true=test_true, y_pred_proba=test_pred)
# Higher is better
auc = roc_auc_score(y_true=test_true, y_score=test_pred)
print("log loss:", lg_loss, '\n' "brier score:", brier, '\n' "auroc:", auc, '\n')

log loss: 0.3049788773059845 
brier score: [0.0960287  0.09685001 0.08729267 0.08290365 0.0662354 ] 
auroc: 0.9236381164414802 



C:\Users\albio\AppData\Local\Temp\ipykernel_2088\2127724836.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lg_loss = loss_function(torch.tensor(test_pred), torch.tensor(test_true)).item()


In [23]:
specificity_array = np.zeros(len(classes))
sensitivity_array = np.zeros(len(classes))
precision_array = np.zeros(len(classes))
accuracy_array = np.zeros(len(classes))

In [24]:

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import scipy.optimize
# Predictions for validation and test
if type(test_pred)!= np.ndarray:
    test_pred = test_pred.detach().cpu().numpy()
if type(test_true)!= np.ndarray:
    test_true = test_true.detach().cpu().numpy()
if type(valid_pred)!= np.ndarray:
    valid_pred = valid_pred.detach().cpu().numpy()
if type(valid_true)!= np.ndarray:
    valid_true = valid_true.detach().cpu().numpy()

test_bin_pred = (test_pred >= 0.5).astype(int)
valid_bin_pred = (valid_pred >= 0.5).astype(int)



# with h5py.File('signal_test_s.h5', 'r') as f:
#     test_true = f['labels'][:]       # or np.array(f['labels'])



# for i in range(10):
#     print(f'{valid_bin_pred[i]} = {valid_true[i]} = {valid_pred[i]}')


# Calculate accuracy on Validation
micro_auroc = roc_auc_score(valid_true, valid_pred, average='micro')
macro_auroc = roc_auc_score(valid_true, valid_pred, average='macro')
micro_f1 = f1_score(valid_true, valid_bin_pred, average='micro')
macro_f1 = f1_score(valid_true, valid_bin_pred, average='macro')
per_class_f1 = f1_score(valid_true, valid_bin_pred, average=None)

print(f"\nOverall Metrics On Validation:")
print(f"Micro Auroc: {micro_auroc:.4f}")
print(f"Macro Auroc: {macro_auroc:.4f}")
print(f"Micro F1-score: {micro_f1:.4f}")
print(f"Macro F1-score: {macro_f1:.4f}")
print(f"Per Class F1-score: {per_class_f1}")

#Calculate accuracy on Test:

# Calculate accuracy on TEST
'''
micro_auroc_test = roc_auc_score(test_true, test_pred, average='micro')
macro_auroc_test = roc_auc_score(test_true, test_pred, average='macro')
micro_f1_test = f1_score(test_true, test_bin_pred, average='micro')
macro_f1_test = f1_score(test_true, test_bin_pred, average='macro')

print(f"\nTest Metrics before Thresholds:")
print(f"Micro Auroc: {micro_auroc_test:.4f}")
print(f"Macro Auroc: {macro_auroc_test:.4f}")
print(f"Micro F1-score: {micro_f1_test:.4f}")
print(f"Macro F1-score: {macro_f1_test:.4f}")
'''

# y_true and y_pred are available from the last evaluation loop on the validation set
n_classes = len(classes)
optimal_thresholds = np.zeros(n_classes)

def f1_threshold(y_true, y_pred, threshold):
    y_pred_bin = (y_pred >= threshold).astype(int)
    return f1_score(y_true, y_pred_bin)

# Search optimal thresholds for each class

_f1_th= lambda th: -f1_threshold(valid_true, valid_pred, th)




th_list = np.linspace(0.01, 0.99, 100)

print(f'\nOptimal thresholds:')
for class_i in range(n_classes):
    best_f1 = 0
    best_th = 0.5
    
    for th in th_list:
        class_predictions = (valid_pred[:, class_i] >= th).astype(int)
        class_f1 = f1_score(valid_true[:, class_i], class_predictions)
        
        if class_f1 > best_f1:
            best_f1 = class_f1
            best_th = th
    
    optimal_thresholds[class_i] = best_th
    print(f'{classes[class_i]}: {best_th:.2f}')


# Predictions on Validation after Thresholds
valid_bin_pred_th= np.zeros_like(valid_bin_pred)
for class_i in range(n_classes):
    valid_bin_pred_th[:, class_i] = (valid_pred[:, class_i] >= optimal_thresholds[class_i]).astype(int)

# Calculate accuracy on Validation after Thresholds
micro_auroc = roc_auc_score(valid_true, valid_pred, average='micro')
macro_auroc = roc_auc_score(valid_true, valid_pred, average='macro')
micro_f1 = f1_score(valid_true, valid_bin_pred_th, average='micro')
macro_f1 = f1_score(valid_true, valid_bin_pred_th, average='macro')
per_class_f1 = f1_score(valid_true, valid_bin_pred_th, average=None)

print(f"\nValidation Metrics after Thresholds:")
print(f"Micro Auroc: {micro_auroc:.4f}")
print(f"Macro Auroc: {macro_auroc:.4f}")
print(f"Micro F1-score: {micro_f1:.4f}")
print(f"Macro F1-score: {macro_f1:.4f}")
print(f"Per Class F1-score: {per_class_f1}")


# for i in range(10):
#     print(f'{valid_bin_pred_th[i]} = {valid_true[i]} = {valid_pred[i]}')

# Predictions on TEST after Thresholds

test_bin_pred_th= np.zeros_like(test_bin_pred)
for class_i in range(n_classes):
    test_bin_pred_th[:, class_i] = (test_pred[:, class_i] >= optimal_thresholds[class_i]).astype(int)

# Calculate accuracy on TEST
micro_auroc_test = roc_auc_score(test_true, test_pred, average='micro')
macro_auroc_test = roc_auc_score(test_true, test_pred, average='macro')
micro_f1_test = f1_score(test_true, test_bin_pred_th, average='micro')
macro_f1_test = f1_score(test_true, test_bin_pred_th, average='macro')
per_class_f1_test = f1_score(test_true, test_bin_pred_th, average=None)

print(f"\nTest Metrics after Thresholds:")
print(f"Micro Auroc: {micro_auroc_test:.4f}")
print(f"Macro Auroc: {macro_auroc_test:.4f}")
print(f"Micro F1-score: {micro_f1_test:.4f}")
print(f"Macro F1-score: {macro_f1_test:.4f}")
print(f"Per Class F1-score: {per_class_f1_test}")

CM= np.zeros((n_classes,4))
for class_i in range(n_classes):
    tn, fp, fn, tp = confusion_matrix(test_true[:, class_i], test_bin_pred_th[:, class_i]).ravel()

    tn, fp, fn, tp = confusion_matrix(test_true[:, class_i], test_bin_pred_th[:, class_i]).ravel()
    CM[class_i,0]= tn
    CM[class_i,1]= fp
    CM[class_i,2]= fn
    CM[class_i,3]= tp

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precission = tp / (tp + fp) if (tp + fp) > 0 else 0
    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    #vs naive metrics
    y_true = test_true[:, class_i]
    n = len(y_true)

    # majority-class naive classifier
    if np.mean(y_true) >= 0.5:
        # always predict positive
        y_pred = np.ones(n, dtype=int)
    else:
        # always predict negative
        y_pred = np.zeros(n, dtype=int)

    naive_tp = np.sum((y_pred == 1) & (y_true == 1))
    naive_tn = np.sum((y_pred == 0) & (y_true == 0))
    naive_fp = np.sum((y_pred == 1) & (y_true == 0))
    naive_fn = np.sum((y_pred == 0) & (y_true == 1))

    naive_accuracy = (naive_tp + naive_tn) / n if n > 0 else 0
    naive_precission = naive_tp / (naive_tp + naive_fp) if (naive_tp + naive_fp) > 0 else 0
    naive_sensitivity = naive_tp / (naive_tp + naive_fn) if (naive_tp + naive_fn) > 0 else 0
    naive_specificity = naive_tn / (naive_tn + naive_fp) if (naive_tn + naive_fp) > 0 else 0
    print(f"{classes[class_i]} - Sensitivity: {sensitivity:.4f}, Specificity: {specificity:.4f}, Precision: {precission:.4f}, Accu: {acc:.4f}")
    print(f"Naive {classes[class_i]} - Sensitivity : {naive_sensitivity:.4f},Specificity: {naive_specificity:.4f}, Precision: {naive_precission:.4f}, Accuracy: {naive_accuracy:.4f}\n")

    #Save in array
    k= 1 #naive, KD, NoKd, Signal
    

    specificity_array[ class_i] = specificity
    sensitivity_array[class_i] = sensitivity
    precision_array[class_i] = precission
    accuracy_array[class_i] = acc

np.savetxt('CM_test.csv', CM, delimiter=',', header='TN, FP, FN, TP', comments='')


Overall Metrics On Validation:
Micro Auroc: 0.9486
Macro Auroc: 0.9418
Micro F1-score: 0.7970
Macro F1-score: 0.7600
Per Class F1-score: [0.87802038 0.77747989 0.7651357  0.77995392 0.59947644]

Optimal thresholds:
NORM: 0.41
MI: 0.43
STTC: 0.48
CD: 0.38
HYP: 0.26

Validation Metrics after Thresholds:
Micro Auroc: 0.9486
Macro Auroc: 0.9418
Micro F1-score: 0.8031
Macro F1-score: 0.7737
Per Class F1-score: [0.88151927 0.78686816 0.76796715 0.78496407 0.64718615]

Test Metrics after Thresholds:
Micro Auroc: 0.9347
Macro Auroc: 0.9236
Micro F1-score: 0.7741
Macro F1-score: 0.7410
Per Class F1-score: [0.8542471  0.73422562 0.75167785 0.760364   0.6042885 ]
NORM - Sensitivity: 0.9190, Specificity: 0.8186, Precision: 0.7980, Accu: 0.8626
Naive NORM - Sensitivity : 0.0000,Specificity: 1.0000, Precision: 0.0000, Accuracy: 0.5619

MI - Sensitivity: 0.6982, Specificity: 0.9320, Precision: 0.7742, Accu: 0.8735
Naive MI - Sensitivity : 0.0000,Specificity: 1.0000, Precision: 0.0000, Accuracy: 0.74

In [25]:
np.savetxt("precision_array.csv", precision_array, delimiter=",")
np.savetxt("sensitivity_array.csv", sensitivity_array, delimiter=",")
np.savetxt("specificity_array.csv", specificity_array, delimiter=",")
np.savetxt("accuracy_array.csv", accuracy_array, delimiter=",")